# Workflow

```
User → LLM → Tool Call → Execute Tool → Send Result → LLM → Final Answer
            (loop until done)
```

This style is used in industry.

In [19]:
!pip install langchain-tavily arxiv


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
            "langchain-tavily", "arxiv", ]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
langchain-tavily version: 0.2.18
arxiv version: 4.0.0


# 1. Tools

In [32]:
import os
from dotenv import load_dotenv
import arxiv 

# 1. Load Environment Variables
load_dotenv()

# Ideally keys should be hidden in .env file
# os.environ["OPENAI_API_KEY"] = "your key"
# os.environ["TAVILY_API_KEY"] = "your_tavily_key"


# 2. Initialize Modern Tools
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch 


search_tool = TavilySearch()

# FIXED: Standardized client lifecycle loop for arxiv 2.0.0+ / 4.0.0+
@tool
def custom_arxiv_search(query: str) -> str:
    """Useful for searching scientific research papers on Arxiv and summarizing them."""
    try:
        # Initialize the modern arxiv Client 
        client = arxiv.Client()
        
        # Build the search parameters safely
        search = arxiv.Search(
            query=str(query),
            max_results=3
        )
        
        # Pull the generator results
        results = client.results(search)
        
        output = []
        # Safely extract from generator loop
        for paper in results:
            output.append(f"Title: {paper.title}\nSummary: {paper.summary}\n")
            
        if not output:
            return "No research papers found for that query."
            
        return "\n".join(output)
    except Exception as e:
        # Returning the error as a string prevents LangGraph's tool node execution from crashing
        return f"Error executing Arxiv search tool: {str(e)}"

# Combine all functioning tools
tools = [ custom_arxiv_search, search_tool ]


# 2. LLM

In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. Create Agent

In [34]:
system_prompt = (
    "You are an expert research assistant. "
    "Use your available tools to look up information and answer the user's questions accurately. "
    "When using custom_arxiv_search, provide a brief keywords query string."
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)


# 4. Run Agent

In [35]:
response1 = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": "Find recent research on transformers in NLP and summarize simply"
                  }]
    }
)

print("Output 1:\n", response1["messages"][-1].content, "\n")

Output 1:
 Here are some recent research papers on transformers in Natural Language Processing (NLP):

1. **Transformers: "The End of History" for NLP?**
   - This paper discusses the significant impact of transformer architectures, like BERT, on NLP tasks. While these models have advanced the field, they have inherent limitations in modeling certain types of information. The authors demonstrate these limitations through practical tasks and datasets, showing that simple modifications can lead to substantial improvements over existing models like RoBERTa and XLNet. They also propose future enhancements to the transformer architecture to increase its expressiveness for NLP applications.

2. **A dissemination workshop for introducing young Italian students to NLP**
   - This research describes a workshop aimed at popularizing NLP among young students in Italy through game-based learning materials. The goal is to engage students and raise awareness about NLP concepts.

3. **Teaching NLP wi

# Some more queries

In [36]:

response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the latest news about AI in India?"}]}
)

print("Output 2:\n", response2["messages"][-1].content, "\n")


Output 2:
 Here are some of the latest news highlights about AI in India:

1. **AI Education Initiatives**: Starting from 2026, AI will be mandatory in schools from Class 3, with efforts to design teaching methods that do not rely on computers. This initiative aims to integrate AI education early in the curriculum. [Read more here](https://www.instagram.com/reel/DZ2uhfwA7U3).

2. **Challenges for AI Startups**: An IIT Hyderabad professor has raised concerns about the potential waste of 10,000 AI startups in India, suggesting that there is a significant vision gap in the country's approach to AI development. [Watch the discussion here](https://www.youtube.com/watch?v=gMo64HsAdAc).

3. **Future of AI Economy**: A discussion titled "ImagiNXT 2026" explores how India can evolve into a trillion-dollar AI economy, focusing on sovereign computing, AI infrastructure, and policy innovation. [Watch the video here](https://www.youtube.com/watch?v=UCcycjo-CGU).

4. **Indigenous AI Solutions**: The